# V8 Final — Leakage-Safe Adversarial Cloud IDS

This notebook has two deliberately separate experimental tracks:

1. **Paper reproduction:** fixed-count variance/MI and SHAP selection, a three-hidden-layer
   DNN, SMOTE, and **static** FGSM + PGD + raw DeepFool augmentation.
2. **V8 extension:** adaptive robust-feature selection and genuine batch-level online PGD
   training using the current model.

The reference methodology is Holla, Polepalli & Sasikumar, *Adversarial Threats to Cloud
IDS: Robust Defense With Adversarial Training and Feature Selection*, IEEE Access (2025).
Its reported numbers are context, not optimization targets.

### Scientific scope

- Paper-specified choices are marked `PAPER-SPECIFIED`.
- Missing details chosen here are marked `IMPLEMENTATION ASSUMPTION`.
- Stronger diagnostics are marked `V8 ADDITIONAL EXPERIMENT`.
- All adversarial examples are feature-space examples. An L∞ constraint after standardization
  does not guarantee physically or protocol-valid network traffic. Functional validation is
  outside scope, consistent with the paper's stated integrity assumption.

In [1]:
# ===== 1. Imports, configuration, reproducibility, and output contract =====
import copy, glob, hashlib, json, os, platform, random, shutil, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             matthews_corrcoef, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
import shap
import sklearn
import art
from art.attacks.evasion import DeepFool
from art.estimators.classification import PyTorchClassifier

warnings.filterwarnings("once")

CONFIG = {
    "experiment_name": "V8_FINAL",
    "data_dir": "data",
    "output_dir": "outputs",
    "seeds": [42, 123, 2024],
    "data_seed": 42,
    "quick_run": False,  # FINAL default; set True only for debugging, never thesis results
    "max_rows_per_file": None,
    "max_train_rows": 500_000,  # IMPLEMENTATION ASSUMPTION: fixed resource-bounded sample
    "test_size": 0.20,
    "development_fraction_of_train": 0.10,
    "validation_samples": 20_000,
    "mi_variance_threshold": 1e-5,
    "mi_sample_size": 50_000,
    "paper_mi_top_t": 40,       # IMPLEMENTATION ASSUMPTION: paper omits T
    "paper_shap_top_t": 15,     # IMPLEMENTATION ASSUMPTION: paper omits T
    "extension_mi_cumulative": 0.70,   # V8 ADDITIONAL EXPERIMENT
    "extension_shap_cumulative": 0.80, # V8 ADDITIONAL EXPERIMENT
    "shap_background_samples": 100,
    "shap_evaluation_samples": 300,
    "smote_sampling_strategy": 1.0,
    "hidden_layers": [128, 64, 32],  # widths are an IMPLEMENTATION ASSUMPTION
    "dropout": 0.30,
    "learning_rate": 0.001,     # PAPER-SPECIFIED
    "batch_size": 2048,
    "baseline_epochs": 50,      # PAPER-SPECIFIED
    "paper_defense_epochs": 50, # PAPER-SPECIFIED
    "extension_clean_epochs": 50,
    "extension_online_epochs": 12,
    "attack_epsilon": 0.05,     # IMPLEMENTATION ASSUMPTION in standardized space
    "pgd_steps": 20,
    "pgd_step_ratio": 0.10,
    "pgd_restarts": 1,
    "deepfool_steps": 50,
    "evaluation_samples": 2000,
    "deepfool_evaluation_samples": 500,
    "square_evaluation_samples": 1000,
    "square_iterations": 300,
    "square_initial_fraction": 0.30,
    "paper_fgsm_train_samples": 20_000,
    "paper_pgd_train_samples": 20_000,
    "paper_deepfool_train_samples": 2_000,
    "online_adversarial_ratio": 0.50,
    "online_pgd_steps": 5,
    "online_clean_loss_weight": 0.50,
    "online_adversarial_loss_weight": 0.50,
    "diagnostic_samples": 500,
    "pgd_strength_steps": [20, 50, 100, 200],
    "diagnostic_restarts": 5,
    "epsilon_sweep": [0.01, 0.025, 0.05, 0.075, 0.10],
    "boundary_samples": 200,
    "boundary_start_trials": 128,
    "boundary_refinement_steps": 20,
    "transfer_samples": 1000,
    "surrogate_epochs": 30,
}
if CONFIG["quick_run"]:
    CONFIG.update(max_train_rows=20_000, baseline_epochs=2, paper_defense_epochs=2,
                  extension_clean_epochs=2, extension_online_epochs=1,
                  evaluation_samples=128, deepfool_evaluation_samples=32,
                  square_evaluation_samples=64, paper_fgsm_train_samples=256,
                  paper_pgd_train_samples=256, paper_deepfool_train_samples=32,
                  diagnostic_samples=32, boundary_samples=16, transfer_samples=64,
                  surrogate_epochs=2, shap_evaluation_samples=32)

OUTPUT_DIR = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "seed_runs").mkdir(exist_ok=True)

def section(title):
    print(f"\n{'=' * 88}\n{title}\n{'=' * 88}", flush=True)

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    try: torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as exc: print(f"Determinism warning: {exc}")

set_seed(CONFIG["data_seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
environment = {
    "python": sys.version, "platform": platform.platform(), "torch": torch.__version__,
    "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__,
    "shap": shap.__version__, "art": getattr(art, "__version__", "unknown"),
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cuda_or_rocm_runtime": getattr(torch.version, "cuda", None) or getattr(torch.version, "hip", None),
    "determinism_note": "GPU kernels may remain nondeterministic; warnings are enabled.",
}
with open(OUTPUT_DIR / "config.json", "w", encoding="utf-8") as f: json.dump(CONFIG, f, indent=2)
with open(OUTPUT_DIR / "environment.json", "w", encoding="utf-8") as f: json.dump(environment, f, indent=2)
print(f"Device={DEVICE} | outputs={OUTPUT_DIR.resolve()} | QUICK_RUN={CONFIG['quick_run']}")

Device=cuda | outputs=F:\PythonProjects\aml_rocm_native\outputs | QUICK_RUN=False


## Leakage-safe data pipeline

Cleaning and duplicate removal occur before splitting. Row splits are then fixed. Categorical
encoding, missing-value imputation, variance filtering, MI, scaling, and SMOTE are fitted using
training rows only. Development data may guide SHAP selection; final test data is evaluation-only.

In [2]:
# ===== 2. Load and clean every available CSE-CIC-IDS2018 file =====
section("DATA — discovery, loading, and cleaning")
t_pipeline_start = time.perf_counter(); t0 = time.perf_counter()
data_dir = Path(CONFIG["data_dir"])
files = sorted(data_dir.rglob("*.parquet")) or sorted(data_dir.rglob("*.csv"))
if not files: raise FileNotFoundError(f"No parquet/CSV files under {data_dir.resolve()}")

frames, file_rows = [], []
for path in tqdm(files, desc="Loading dataset files"):
    frame = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path, low_memory=False)
    if CONFIG["max_rows_per_file"] and len(frame) > CONFIG["max_rows_per_file"]:
        frame = frame.sample(CONFIG["max_rows_per_file"], random_state=CONFIG["data_seed"])
    frame.columns = frame.columns.astype(str).str.strip()
    frame = frame.loc[:, ~frame.columns.duplicated()].copy()
    file_rows.append({"file": path.name, "rows_loaded": len(frame)})
    frames.append(frame)
raw = pd.concat(frames, ignore_index=True); del frames

label_candidates = [c for c in raw.columns if c.lower() == "label"]
if len(label_candidates) != 1: raise ValueError(f"Expected one Label column, found {label_candidates}")
raw = raw.rename(columns={label_candidates[0]: "Label"})
raw = raw[raw["Label"].astype(str).str.strip().str.lower().ne("label")].copy()

identity_names = {"timestamp", "flow id", "src ip", "dst ip", "source ip", "destination ip",
                  "src ip addr", "dst ip addr"}
identity_columns = [c for c in raw.columns if c.strip().lower() in identity_names]
raw.drop(columns=identity_columns, inplace=True, errors="ignore")
before_duplicates = len(raw); raw.drop_duplicates(inplace=True)
raw["target"] = raw["Label"].astype(str).str.strip().str.lower().ne("benign").astype(np.int64)
raw.drop(columns="Label", inplace=True)
if raw["target"].nunique() != 2: raise ValueError("Binary target construction did not produce both classes")

# Split row identities before learning any preprocessing statistic.
all_indices = np.arange(len(raw))
train_pool_idx, test_idx = train_test_split(
    all_indices, test_size=CONFIG["test_size"], stratify=raw["target"].to_numpy(),
    random_state=CONFIG["data_seed"])
train_idx, dev_idx = train_test_split(
    train_pool_idx, test_size=CONFIG["development_fraction_of_train"],
    stratify=raw.iloc[train_pool_idx]["target"].to_numpy(), random_state=CONFIG["data_seed"] + 1)
split_owner = np.zeros(len(raw), dtype=np.uint8)
for split_code, split_indices in enumerate((train_idx, dev_idx, test_idx), start=1):
    if np.any(split_owner[split_indices] != 0): raise AssertionError("Split overlap detected")
    split_owner[split_indices] = split_code
del split_owner

def stratified_cap(indices, limit, seed):
    if limit is None or len(indices) <= limit: return np.asarray(indices)
    kept, _ = train_test_split(indices, train_size=limit, stratify=raw.iloc[indices]["target"], random_state=seed)
    return np.asarray(kept)

train_idx = stratified_cap(train_idx, CONFIG["max_train_rows"], CONFIG["data_seed"])
X_train_df = raw.iloc[train_idx].drop(columns="target").reset_index(drop=True)
X_dev_df = raw.iloc[dev_idx].drop(columns="target").reset_index(drop=True)
X_test_df = raw.iloc[test_idx].drop(columns="target").reset_index(drop=True)
y_train = raw.iloc[train_idx]["target"].to_numpy(np.int64)
y_dev = raw.iloc[dev_idx]["target"].to_numpy(np.int64)
y_test = raw.iloc[test_idx]["target"].to_numpy(np.int64)

data_summary = pd.DataFrame([
    {"item": "files", "value": len(files)}, {"item": "raw_rows", "value": before_duplicates},
    {"item": "duplicate_rows_removed", "value": before_duplicates - len(raw)},
    {"item": "identity_columns_removed", "value": json.dumps(identity_columns)},
    {"item": "train_rows_used", "value": len(train_idx)}, {"item": "development_rows", "value": len(dev_idx)},
    {"item": "test_rows", "value": len(test_idx)}, {"item": "benign_total", "value": int((raw.target == 0).sum())},
    {"item": "attack_total", "value": int((raw.target == 1).sum())},
    {"item": "files_loaded", "value": json.dumps([p.name for p in files])},
])
data_summary.to_csv(OUTPUT_DIR / "data_summary.csv", index=False)
preprocessing_time_load_clean_split = time.perf_counter() - t0
print(data_summary.to_string(index=False)); del raw


DATA — discovery, loading, and cleaning


Loading dataset files:   0%|          | 0/10 [00:00<?, ?it/s]

                    item                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  value
                   files                                                                                                                                                                                                                                                                                                                                

In [3]:
# ===== 3. Training-fitted encoding, imputation, fixed MI selection, scaling, and SMOTE =====
section("FEATURE PIPELINE — train-only fit")
t0 = time.perf_counter()

# Protocol is treated categorically as described by the paper. Other nonnumeric columns are also categorical.
protocol_cols = [c for c in X_train_df.columns if c.strip().lower() == "protocol"]
object_cols = X_train_df.select_dtypes(include=["object", "category", "string"]).columns.tolist()
categorical_cols = sorted(set(protocol_cols + object_cols))
numeric_cols = [c for c in X_train_df.columns if c not in categorical_cols]

for frame in (X_train_df, X_dev_df, X_test_df):
    for col in numeric_cols:
        frame[col] = pd.to_numeric(frame[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

train_medians = X_train_df[numeric_cols].median().fillna(0.0)
def numeric_matrix(frame):
    return frame[numeric_cols].fillna(train_medians).to_numpy(np.float32)

try: encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
except TypeError: encoder = OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=np.float32)
if categorical_cols:
    def categorical_frame(frame):
        values = frame[categorical_cols].astype(object)
        return values.where(values.notna(), "<MISSING>").astype(str)
    encoder.fit(categorical_frame(X_train_df))
    encoded_names = encoder.get_feature_names_out(categorical_cols).tolist()
    def encode(frame):
        return np.hstack([numeric_matrix(frame), encoder.transform(categorical_frame(frame))])
else:
    encoded_names = []
    def encode(frame): return numeric_matrix(frame)

feature_names_all = numeric_cols + encoded_names
X_train_encoded, X_dev_encoded, X_test_encoded = encode(X_train_df), encode(X_dev_df), encode(X_test_df)
if not all(np.isfinite(x).all() for x in (X_train_encoded, X_dev_encoded, X_test_encoded)):
    raise AssertionError("Non-finite values remain after training-fitted imputation")

variance = VarianceThreshold(CONFIG["mi_variance_threshold"])
X_train_var = variance.fit_transform(X_train_encoded)
X_dev_var, X_test_var = variance.transform(X_dev_encoded), variance.transform(X_test_encoded)
variance_names = np.asarray(feature_names_all)[variance.get_support()].tolist()

rng = np.random.RandomState(CONFIG["data_seed"])
mi_n = min(CONFIG["mi_sample_size"], len(X_train_var))
mi_idx = rng.choice(len(X_train_var), mi_n, replace=False)
mi_values = mutual_info_classif(X_train_var[mi_idx], y_train[mi_idx], random_state=CONFIG["data_seed"])
mi_scores = pd.DataFrame({"feature": variance_names, "mi_score": mi_values}).sort_values("mi_score", ascending=False)
mi_scores["rank"] = np.arange(1, len(mi_scores) + 1)
mi_scores.to_csv(OUTPUT_DIR / "mi_scores.csv", index=False)

paper_mi_features = mi_scores.head(min(CONFIG["paper_mi_top_t"], len(mi_scores)))["feature"].tolist()
if len(paper_mi_features) < 2: raise AssertionError("Too few paper MI features")
paper_mi_positions = [variance_names.index(f) for f in paper_mi_features]
X_train_mi_raw, X_dev_mi_raw, X_test_mi_raw = (X_train_var[:, paper_mi_positions],
                                                X_dev_var[:, paper_mi_positions],
                                                X_test_var[:, paper_mi_positions])
scaler = StandardScaler().fit(X_train_mi_raw)
X_train_mi = scaler.transform(X_train_mi_raw).astype(np.float32)
X_dev_mi = scaler.transform(X_dev_mi_raw).astype(np.float32)
X_test_mi = scaler.transform(X_test_mi_raw).astype(np.float32)
clip_min, clip_max = X_train_mi.min(0), X_train_mi.max(0)
if np.any(clip_min >= clip_max): raise AssertionError("Selected feature with zero training range")

smote = SMOTE(random_state=CONFIG["data_seed"], sampling_strategy=CONFIG["smote_sampling_strategy"])
X_train_bal, y_train_bal = smote.fit_resample(X_train_mi, y_train)
X_train_bal = np.ascontiguousarray(X_train_bal, dtype=np.float32)
y_train_bal = np.asarray(y_train_bal, dtype=np.int64)

structural_mask = np.ones(len(paper_mi_features), dtype=np.float32)
for i, name in enumerate(paper_mi_features):
    if any(name == c or name.startswith(c + "_") for c in categorical_cols): structural_mask[i] = 0.0
if structural_mask.sum() == 0: raise AssertionError("No mutable continuous features remain")

with open(OUTPUT_DIR / "selected_features_mi.json", "w", encoding="utf-8") as f:
    json.dump({"paper_fixed_top_t": CONFIG["paper_mi_top_t"], "features": paper_mi_features}, f, indent=2)
preprocessing_time = preprocessing_time_load_clean_split + (time.perf_counter() - t0)
print(f"Encoded={len(feature_names_all)} | variance-kept={len(variance_names)} | "
      f"paper MI={len(paper_mi_features)} | train={X_train_mi.shape} | SMOTE={X_train_bal.shape}")
del X_train_df, X_dev_df, X_test_df, X_train_encoded, X_dev_encoded, X_test_encoded
del X_train_var, X_dev_var, X_test_var, X_train_mi_raw, X_dev_mi_raw, X_test_mi_raw


FEATURE PIPELINE — train-only fit
Encoded=79 | variance-kept=71 | paper MI=40 | train=(500000, 40) | SMOTE=(790214, 40)


In [4]:
# ===== 4. One authoritative model, training, prediction, and metric implementation =====
section("CORE MODEL AND TRAINING HELPERS")

class IDSNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        h1, h2, h3 = CONFIG["hidden_layers"]
        self.network = nn.Sequential(
            nn.Linear(input_dim, h1), nn.BatchNorm1d(h1), nn.ReLU(), nn.Dropout(CONFIG["dropout"]),
            nn.Linear(h1, h2), nn.BatchNorm1d(h2), nn.ReLU(), nn.Dropout(CONFIG["dropout"]),
            nn.Linear(h2, h3), nn.BatchNorm1d(h3), nn.ReLU(), nn.Dropout(CONFIG["dropout"]),
            nn.Linear(h3, 2))
    def forward(self, x): return self.network(x)

def new_model(input_dim, seed):
    # Seed before construction so initialization is genuinely controlled.
    set_seed(seed)
    return IDSNet(input_dim)

def make_loader(X, y, batch_size, shuffle, seed):
    generator = torch.Generator(); generator.manual_seed(seed)
    dataset = TensorDataset(torch.as_tensor(X, dtype=torch.float32), torch.as_tensor(y, dtype=torch.long))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator,
                      num_workers=0, pin_memory=torch.cuda.is_available())

def predict_logits(model, X, device=DEVICE, batch_size=8192):
    model.eval(); chunks = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb = torch.as_tensor(X[start:start + batch_size], dtype=torch.float32, device=device)
            chunks.append(model(xb).cpu().numpy())
    return np.concatenate(chunks) if chunks else np.empty((0, 2), np.float32)

def classification_metrics(model, X, y, threshold=0.5):
    logits = predict_logits(model, X); prob = torch.softmax(torch.from_numpy(logits), 1).numpy()[:, 1]
    pred = (prob >= threshold).astype(np.int64)
    return {"n": len(y), "accuracy": accuracy_score(y, pred),
            "balanced_accuracy": balanced_accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0),
            "mcc": matthews_corrcoef(y, pred),
            "roc_auc": roc_auc_score(y, prob) if len(np.unique(y)) == 2 else np.nan}

def train_clean(model, X, y, X_val, y_val, epochs, seed, description):
    set_seed(seed); model.to(DEVICE)
    loader = make_loader(X, y, CONFIG["batch_size"], True, seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
    history = []
    val_idx = np.random.RandomState(CONFIG["data_seed"] + 77).choice(
        len(X_val), min(CONFIG["validation_samples"], len(X_val)), replace=False)
    for epoch in tqdm(range(epochs), desc=description):
        model.train(); loss_sum = 0.0; seen = 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(set_to_none=True); loss = F.cross_entropy(model(xb), yb)
            loss.backward(); optimizer.step(); loss_sum += loss.item() * len(xb); seen += len(xb)
        val = classification_metrics(model, X_val[val_idx], y_val[val_idx])
        history.append({"epoch": epoch + 1, "train_loss": loss_sum / seen,
                        "val_accuracy": val["accuracy"], "val_f1": val["f1"]})
        if (epoch + 1) % 5 == 0 or epoch == 0: print(f"{description}: epoch={epoch+1}/{epochs} val_f1={val['f1']:.4f}", flush=True)
    return pd.DataFrame(history)

def model_checksum(model):
    h = hashlib.sha256()
    for p in model.parameters(): h.update(p.detach().cpu().numpy().tobytes())
    return h.hexdigest()


CORE MODEL AND TRAINING HELPERS


In [5]:
# ===== 5. Authoritative constrained attacks and attack sanity statistics =====
section("ATTACK IMPLEMENTATIONS AND VALIDATION")

def _bounds_t(bounds, device): return torch.as_tensor(bounds, dtype=torch.float32, device=device)

def fgsm_attack(model, X, y, eps, bounds, mask):
    model.eval(); x0 = torch.as_tensor(X, dtype=torch.float32, device=DEVICE)
    yt = torch.as_tensor(y, dtype=torch.long, device=DEVICE); x = x0.clone().requires_grad_(True)
    loss = F.cross_entropy(model(x), yt); grad = torch.autograd.grad(loss, x)[0]
    lo, hi, mt = _bounds_t(bounds[0], DEVICE), _bounds_t(bounds[1], DEVICE), _bounds_t(mask, DEVICE)
    # A test value may lie outside training extrema. Expand only enough to include its clean value.
    domain_lo, domain_hi = torch.minimum(lo, x0), torch.maximum(hi, x0)
    adv = x0 + eps * grad.sign() * mt
    adv = torch.maximum(torch.minimum(adv, domain_hi), domain_lo)
    adv = x0 + (adv - x0) * mt
    return adv.detach().cpu().numpy().astype(np.float32)

def pgd_attack(model, X, y, eps, steps, restarts, bounds, mask, seed, random_start=True):
    model.eval(); x0 = torch.as_tensor(X, dtype=torch.float32, device=DEVICE)
    yt = torch.as_tensor(y, dtype=torch.long, device=DEVICE)
    lo, hi, mt = _bounds_t(bounds[0], DEVICE), _bounds_t(bounds[1], DEVICE), _bounds_t(mask, DEVICE)
    domain_lo, domain_hi = torch.minimum(lo, x0), torch.maximum(hi, x0)
    alpha = eps * CONFIG["pgd_step_ratio"]; best, best_loss = x0.clone(), torch.full((len(X),), -torch.inf, device=DEVICE)
    for restart in range(restarts):
        generator = torch.Generator(device=DEVICE); generator.manual_seed(seed + restart)
        delta = torch.empty_like(x0).uniform_(-eps, eps, generator=generator) * mt if random_start else torch.zeros_like(x0)
        x = torch.maximum(torch.minimum(x0 + delta, domain_hi), domain_lo).detach()
        for _ in range(steps):
            x.requires_grad_(True); losses = F.cross_entropy(model(x), yt, reduction="none")
            grad = torch.autograd.grad(losses.sum(), x)[0]
            x = x.detach() + alpha * grad.sign() * mt
            x = torch.maximum(torch.minimum(x, x0 + eps), x0 - eps)
            x = torch.maximum(torch.minimum(x, domain_hi), domain_lo)
            x = (x0 + (x - x0) * mt).detach()
        with torch.no_grad(): losses = F.cross_entropy(model(x), yt, reduction="none")
        take = losses > best_loss; best[take] = x[take]; best_loss[take] = losses[take]
    return best.cpu().numpy().astype(np.float32)

def make_art_classifier(model, input_dim, bounds):
    cpu_model = IDSNet(input_dim); cpu_model.load_state_dict(copy.deepcopy(model.state_dict())); cpu_model.eval()
    optimizer = torch.optim.Adam(cpu_model.parameters(), lr=CONFIG["learning_rate"])
    # Raw DeepFool is intentionally unconstrained; no L_inf projection or training-range clip is applied.
    return PyTorchClassifier(model=cpu_model, loss=nn.CrossEntropyLoss(), optimizer=optimizer,
                             input_shape=(input_dim,), nb_classes=2, clip_values=None, device_type="cpu")

def raw_deepfool_attack(model, X, bounds):
    classifier = make_art_classifier(model, X.shape[1], bounds)
    attack = DeepFool(classifier=classifier, max_iter=CONFIG["deepfool_steps"])
    return attack.generate(x=np.asarray(X, np.float32)).astype(np.float32)

def square_style_attack(model, X, y, eps, bounds, mask, seed):
    # Score-based tabular random search inspired by Square Attack; not canonical Square Attack.
    rng = np.random.RandomState(seed); clean = np.asarray(X, np.float32); adv = clean.copy()
    mutable = np.flatnonzero(mask); queries = np.ones(len(X), dtype=np.int64)
    clean_prob = torch.softmax(torch.from_numpy(predict_logits(model, clean)), 1).numpy()
    best = clean_prob[np.arange(len(y)), y]; active = clean_prob.argmax(1) == y
    for iteration in tqdm(range(CONFIG["square_iterations"]), desc="Square-style", leave=False):
        ids = np.flatnonzero(active)
        if not len(ids): break
        fraction = max(CONFIG["square_initial_fraction"] * (1 - iteration / CONFIG["square_iterations"]), 1 / len(mutable))
        n_features = max(1, min(len(mutable), int(np.ceil(fraction * len(mutable)))))
        candidate = adv[ids].copy()
        for row in range(len(ids)):
            chosen = rng.choice(mutable, n_features, replace=False)
            candidate[row, chosen] += rng.choice([-eps, eps], n_features)
        candidate = np.clip(candidate, clean[ids] - eps, clean[ids] + eps)
        row_lo, row_hi = np.minimum(bounds[0], clean[ids]), np.maximum(bounds[1], clean[ids])
        candidate = np.maximum(np.minimum(candidate, row_hi), row_lo).astype(np.float32)
        candidate[:, mask == 0] = clean[ids][:, mask == 0]
        prob = torch.softmax(torch.from_numpy(predict_logits(model, candidate)), 1).numpy(); queries[ids] += 1
        score = prob[np.arange(len(ids)), y[ids]]; improve = score < best[ids]; accepted = ids[improve]
        adv[accepted] = candidate[improve]; best[accepted] = score[improve]
        active[accepted[prob[improve].argmax(1) != y[accepted]]] = False
    return adv, queries

def attack_sanity(model, X_clean, X_adv, y, attack, eps=None, queries=None):
    if not np.isfinite(X_adv).all(): raise RuntimeError(f"{attack} produced non-finite values")
    delta = np.asarray(X_adv, np.float64) - np.asarray(X_clean, np.float64)
    abs_delta = np.abs(delta); linf = abs_delta.max(1); changed = linf > 1e-7
    clean_pred = predict_logits(model, X_clean).argmax(1); adv_pred = predict_logits(model, X_adv).argmax(1)
    initially_correct = clean_pred == y
    success = (adv_pred != y) & initially_correct
    result = {"attack": attack, "generated": len(y), "changed_n": int(changed.sum()),
              "changed_pct": 100 * changed.mean(), "mean_l0": (abs_delta > 1e-7).sum(1).mean(),
              "mean_l2": np.linalg.norm(delta, axis=1).mean(), "mean_linf": linf.mean(),
              "max_linf": linf.max(), "within_epsilon_pct": np.nan if eps is None else 100 * (linf <= eps + 1e-5).mean(),
              "attack_success_rate": success.sum() / max(initially_correct.sum(), 1),
              "clean_accuracy_subset": (clean_pred == y).mean(), "adversarial_accuracy": (adv_pred == y).mean(),
              "mean_queries": np.nan if queries is None else np.mean(queries)}
    if eps is not None and result["within_epsilon_pct"] < 100: raise AssertionError(f"{attack} exceeded epsilon")
    if attack in {"FGSM", "PGD"} and result["changed_pct"] < 50: raise RuntimeError(f"{attack} mostly unchanged; aborting")
    if result["changed_pct"] < 95: print(f"WARNING: {attack} changed only {result['changed_pct']:.1f}% of samples")
    print(f"{attack} SANITY | generated={len(y)} changed={result['changed_pct']:.1f}% "
          f"meanL0={result['mean_l0']:.2f} meanL2={result['mean_l2']:.4f} "
          f"meanLinf={result['mean_linf']:.4f} maxLinf={result['max_linf']:.4f} "
          f"within={result['within_epsilon_pct']} ASR={result['attack_success_rate']:.4f}")
    return result


ATTACK IMPLEMENTATIONS AND VALIDATION


# A. PAPER REPRODUCTION

**Paper-specified:** variance/MI selection, SHAP stability, SMOTE, 3 hidden layers with
BatchNorm and Dropout, Adam at 0.001 for 50 epochs, and static adversarial augmentation.

**Implementation assumptions:** all ten locally supplied dataset files; fixed MI T=40 and
SHAP T=15; layer widths 128/64/32; epsilon 0.05 in standardized space; PGD settings; seeds;
SMOTE ratio; and bounded attack-generation sample counts. The static adversarial set is generated
once with a declared seed-42 generator and held fixed across model-initialization seeds. Raw
DeepFool is not projected into an L∞ ball and is always reported separately.

In [6]:
# ===== 6. Paper feature-selection reproduction: fixed T_MI then fixed T_SHAP =====
section("PAPER REPRODUCTION — fixed MI + SHAP selection")
t0 = time.perf_counter()
selector_model = new_model(len(paper_mi_features), CONFIG["seeds"][0])
selector_training_started = time.perf_counter()
selector_history = train_clean(selector_model, X_train_bal, y_train_bal, X_dev_mi, y_dev,
                               CONFIG["baseline_epochs"], CONFIG["seeds"][0], "Paper selector baseline")
selector_training_seconds = time.perf_counter() - selector_training_started

rng = np.random.RandomState(CONFIG["data_seed"] + 9)
shap_idx = rng.choice(len(X_dev_mi), min(CONFIG["shap_evaluation_samples"], len(X_dev_mi)), replace=False)
background_idx = rng.choice(len(X_train_mi), min(CONFIG["shap_background_samples"], len(X_train_mi)), replace=False)
X_shap_clean, y_shap = X_dev_mi[shap_idx], y_dev[shap_idx]
X_shap_adv = pgd_attack(selector_model, X_shap_clean, y_shap, CONFIG["attack_epsilon"],
                        CONFIG["pgd_steps"], 1, (clip_min, clip_max), structural_mask,
                        CONFIG["data_seed"] + 10)
shap_development_stats = attack_sanity(
    selector_model, X_shap_clean, X_shap_adv, y_shap, "SHAP development PGD", CONFIG["attack_epsilon"])
shap_development_stats.update({"method": "Feature Selection", "seed": CONFIG["seeds"][0], "phase": "development_shap"})

background_t = torch.as_tensor(X_train_mi[background_idx], dtype=torch.float32, device=DEVICE)
selector_model.eval(); explainer = shap.GradientExplainer(selector_model, background_t)
def positive_class_shap(values):
    if isinstance(values, list): return np.asarray(values[1])
    values = np.asarray(values)
    if values.ndim == 3 and values.shape[-1] == 2: return values[..., 1]
    if values.ndim == 3 and values.shape[0] == 2: return values[1]
    if values.ndim == 2: return values
    raise ValueError(f"Unsupported SHAP output shape: {values.shape}")

shap_clean_values = positive_class_shap(explainer.shap_values(torch.as_tensor(X_shap_clean, dtype=torch.float32, device=DEVICE)))
shap_adversarial_values = positive_class_shap(explainer.shap_values(torch.as_tensor(X_shap_adv, dtype=torch.float32, device=DEVICE)))
if shap_clean_values.shape != shap_adversarial_values.shape or shap_clean_values.shape[1] != len(paper_mi_features):
    raise AssertionError("SHAP clean/adversarial feature spaces differ")
clean_importance = np.mean(np.abs(shap_clean_values), axis=0)
adversarial_importance = np.mean(np.abs(shap_adversarial_values), axis=0)
paper_stability_raw = 1.0 - np.abs(clean_importance - adversarial_importance) # paper equation
paper_stability = np.clip(paper_stability_raw, 0.0, 1.0) # numerical interpretation stated explicitly
mean_importance = 0.5 * (clean_importance + adversarial_importance)
relative_shift = np.abs(clean_importance - adversarial_importance) / (mean_importance + 1e-12)
extension_stability = 1.0 - np.clip(relative_shift, 0.0, 1.0)
extension_robust_score = mean_importance * extension_stability

shap_scores = pd.DataFrame({"feature": paper_mi_features, "clean_importance": clean_importance,
                            "adversarial_importance": adversarial_importance,
                            "paper_stability_raw": paper_stability_raw, "paper_stability_clipped": paper_stability,
                            "extension_relative_stability": extension_stability,
                            "extension_robust_score": extension_robust_score})
shap_scores.sort_values("paper_stability_clipped", ascending=False).to_csv(OUTPUT_DIR / "shap_stability_scores.csv", index=False)
np.savez_compressed(OUTPUT_DIR / "shap_values_clean_adversarial.npz",
                    clean=shap_clean_values, adversarial=shap_adversarial_values, row_indices=shap_idx)

paper_shap_features = shap_scores.sort_values(["paper_stability_clipped", "clean_importance"], ascending=False).head(
    min(CONFIG["paper_shap_top_t"], len(shap_scores)))["feature"].tolist()
paper_idx = [paper_mi_features.index(f) for f in paper_shap_features]
with open(OUTPUT_DIR / "selected_features_shap.json", "w", encoding="utf-8") as f:
    json.dump({"paper_fixed_top_t": CONFIG["paper_shap_top_t"], "features": paper_shap_features}, f, indent=2)

positive_mi = mi_scores[mi_scores.feature.isin(paper_mi_features)].set_index("feature").loc[paper_mi_features, "mi_score"].clip(lower=0)
cumulative_mi = positive_mi / max(positive_mi.sum(), 1e-12)
n_ext_mi = max(2, int((cumulative_mi.cumsum() < CONFIG["extension_mi_cumulative"]).sum()) + 1)
extension_mi_features = paper_mi_features[:n_ext_mi]
ext_candidates = shap_scores[shap_scores.feature.isin(extension_mi_features)].sort_values("extension_robust_score", ascending=False)
positive_ext = ext_candidates.extension_robust_score.clip(lower=0)
n_ext = max(2, int(((positive_ext.cumsum() / max(positive_ext.sum(), 1e-12)) < CONFIG["extension_shap_cumulative"]).sum()) + 1)
extension_features = ext_candidates.head(n_ext).feature.tolist(); extension_idx = [paper_mi_features.index(f) for f in extension_features]
with open(OUTPUT_DIR / "selected_features_extension.json", "w", encoding="utf-8") as f:
    json.dump({"method": "adaptive cumulative robust SHAP score", "features": extension_features}, f, indent=2)
feature_selection_time = time.perf_counter() - t0
print(f"Paper fixed SHAP={len(paper_shap_features)} | V8 adaptive features={len(extension_features)}")


PAPER REPRODUCTION — fixed MI + SHAP selection


Paper selector baseline:   0%|          | 0/50 [00:00<?, ?it/s]

F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:33.)
  return data.pin_memory(device)


Paper selector baseline: epoch=1/50 val_f1=0.9004


F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:33.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm

...

Paper selector baseline: epoch=40/50 val_f1=0.9187


F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:33.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm

Paper selector baseline: epoch=45/50 val_f1=0.9253


F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:33.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm

Paper selector baseline: epoch=50/50 val_f1=0.9182
SHAP development PGD SANITY | generated=300 changed=100.0% meanL0=37.09 meanL2=0.2452 meanLinf=0.0500 maxLinf=0.0500 within=100.0 ASR=0.1345
Paper fixed SHAP=15 | V8 adaptive features=12


In [7]:
# ===== 7. Paper-style STATIC adversarial augmentation (never called online training) =====
section("PAPER REPRODUCTION — static FGSM + PGD + raw DeepFool augmentation")
X_train_paper, X_dev_paper, X_test_paper = X_train_bal[:, paper_idx], X_dev_mi[:, paper_idx], X_test_mi[:, paper_idx]
paper_bounds = (clip_min[paper_idx], clip_max[paper_idx]); paper_mask = structural_mask[paper_idx]

paper_generator = new_model(len(paper_idx), CONFIG["seeds"][0] + 700)
paper_generator_started = time.perf_counter()
paper_generator_history = train_clean(paper_generator, X_train_paper, y_train_bal, X_dev_paper, y_dev,
                                      CONFIG["baseline_epochs"], CONFIG["seeds"][0] + 700,
                                      "Paper static-attack generator")
paper_generator_training_seconds = time.perf_counter() - paper_generator_started
rng = np.random.RandomState(CONFIG["data_seed"] + 31)
def choose_train(n): return rng.choice(len(X_train_paper), min(n, len(X_train_paper)), replace=False)

t0 = time.perf_counter(); static_parts, static_labels, static_sanity = [], [], []
for attack_name, n_samples in [("FGSM", CONFIG["paper_fgsm_train_samples"]),
                               ("PGD", CONFIG["paper_pgd_train_samples"]),
                               ("DeepFool_raw", CONFIG["paper_deepfool_train_samples"])]:
    ids = choose_train(n_samples); clean, labels = X_train_paper[ids], y_train_bal[ids]
    if attack_name == "FGSM": adv = fgsm_attack(paper_generator, clean, labels, CONFIG["attack_epsilon"], paper_bounds, paper_mask); eps = CONFIG["attack_epsilon"]
    elif attack_name == "PGD": adv = pgd_attack(paper_generator, clean, labels, CONFIG["attack_epsilon"], CONFIG["pgd_steps"], 1, paper_bounds, paper_mask, CONFIG["data_seed"] + 32); eps = CONFIG["attack_epsilon"]
    else: adv = raw_deepfool_attack(paper_generator, clean, paper_bounds); eps = None
    stats = attack_sanity(paper_generator, clean, adv, labels, f"paper_train_{attack_name}", eps)
    stats.update({"method": "Paper Defense", "seed": CONFIG["seeds"][0], "phase": "static_training_generation"})
    changed = np.abs(adv - clean).max(1) > 1e-7
    if not changed.any(): raise RuntimeError(f"{attack_name} produced no changed static-training examples")
    stats["changed_examples_appended"] = int(changed.sum())
    static_sanity.append(stats); static_parts.append(adv[changed]); static_labels.append(labels[changed])
X_paper_static = np.concatenate([X_train_paper] + static_parts).astype(np.float32)
y_paper_static = np.concatenate([y_train_bal] + static_labels).astype(np.int64)
paper_attack_generation_time = time.perf_counter() - t0
print(f"Static clean={len(X_train_paper):,} adversarial={sum(map(len, static_parts)):,} total={len(X_paper_static):,}")


PAPER REPRODUCTION — static FGSM + PGD + raw DeepFool augmentation


Paper static-attack generator:   0%|          | 0/50 [00:00<?, ?it/s]

F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:33.)
  return data.pin_memory(device)


Paper static-attack generator: epoch=1/50 val_f1=0.8411


F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:33.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm

...

Paper static-attack generator: epoch=45/50 val_f1=0.8535


F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:33.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm

Paper static-attack generator: epoch=50/50 val_f1=0.8564
paper_train_FGSM SANITY | generated=20000 changed=100.0% meanL0=13.64 meanL2=0.1552 meanLinf=0.0500 maxLinf=0.0500 within=100.0 ASR=0.3316
paper_train_PGD SANITY | generated=20000 changed=100.0% meanL0=13.85 meanL2=0.1360 meanLinf=0.0498 maxLinf=0.0500 within=100.0 ASR=0.3461


DeepFool:   0%|          | 0/2000 [00:00<?, ?it/s]

paper_train_DeepFool_raw SANITY | generated=2000 changed=100.0% meanL0=15.00 meanL2=1.4800 meanLinf=0.8812 maxLinf=76.9080 within=nan ASR=0.9376
Static clean=790,214 adversarial=41,998 total=832,212


# B. V8 EXTENSION

The extension uses the adaptive robust-SHAP subset and true on-the-fly PGD. Every batch creates
new examples from the **current** model before the optimizer update. This has a different update
and compute budget from the paper reproduction, so performance and overhead are reported rather
than treated as an isolated proof of superiority.

In [8]:
# ===== 8. Genuine online PGD training using current weights =====
section("V8 EXTENSION — online PGD implementation")

def train_online_pgd(model, X, y, X_val, y_val, bounds, mask, seed):
    set_seed(seed); model.to(DEVICE); loader = make_loader(X, y, CONFIG["batch_size"], True, seed + 1)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"]); history = []
    probe_n = min(32, len(X)); probe_x, probe_y = X[:probe_n], y[:probe_n]; previous_probe = None
    val_idx = np.random.RandomState(CONFIG["data_seed"] + 77).choice(
        len(X_val), min(CONFIG["validation_samples"], len(X_val)), replace=False)
    for epoch in tqdm(range(CONFIG["extension_online_epochs"]), desc="V8 online PGD"):
        checksum_before = model_checksum(model); clean_sum = adv_sum = total_sum = perturb_sum = 0.0; seen = adv_seen = 0
        for batch_number, (xb_cpu, yb_cpu) in enumerate(loader):
            xb, yb = xb_cpu.to(DEVICE), yb_cpu.to(DEVICE); n_adv = max(1, int(len(xb) * CONFIG["online_adversarial_ratio"]))
            # Current weights are used here, inside every batch; generated tensors are immediately discarded.
            adv_np = pgd_attack(model, xb[:n_adv].detach().cpu().numpy(), yb[:n_adv].cpu().numpy(),
                                CONFIG["attack_epsilon"], CONFIG["online_pgd_steps"], 1, bounds, mask,
                                seed + epoch * 1_000_003 + batch_number)
            x_adv = torch.as_tensor(adv_np, dtype=torch.float32, device=DEVICE)
            model.train(); optimizer.zero_grad(set_to_none=True)
            clean_loss = F.cross_entropy(model(xb), yb); adv_loss = F.cross_entropy(model(x_adv), yb[:n_adv])
            total_loss = CONFIG["online_clean_loss_weight"] * clean_loss + CONFIG["online_adversarial_loss_weight"] * adv_loss
            total_loss.backward(); optimizer.step()
            clean_sum += clean_loss.item() * len(xb); adv_sum += adv_loss.item() * n_adv
            total_sum += total_loss.item() * len(xb); seen += len(xb); adv_seen += n_adv
            perturb_sum += np.abs(adv_np - xb[:n_adv].detach().cpu().numpy()).max(1).sum()
        probe_adv = pgd_attack(model, probe_x, probe_y, CONFIG["attack_epsilon"], CONFIG["online_pgd_steps"], 1,
                               bounds, mask, seed + 9_000_000 + epoch)
        regenerated = previous_probe is None or not np.array_equal(previous_probe, probe_adv)
        if previous_probe is not None and not regenerated: raise RuntimeError("Online adversarial probe was identical across epochs")
        previous_probe = probe_adv.copy(); checksum_after = model_checksum(model)
        if checksum_before == checksum_after: raise RuntimeError("Model weights did not update during online training")
        val = classification_metrics(model, X_val[val_idx], y_val[val_idx])
        history.append({"epoch": epoch + 1, "clean_samples": seen, "adversarial_samples": adv_seen,
                        "pgd_steps": CONFIG["online_pgd_steps"], "epsilon": CONFIG["attack_epsilon"],
                        "mean_linf": perturb_sum / adv_seen, "clean_loss": clean_sum / seen,
                        "adversarial_loss": adv_sum / adv_seen, "total_loss": total_sum / seen,
                        "val_accuracy": val["accuracy"], "val_f1": val["f1"],
                        "attacks_regenerated": regenerated, "weights_changed": checksum_before != checksum_after})
        print(f"online epoch={epoch+1} clean={seen:,} adv={adv_seen:,} meanLinf={perturb_sum/adv_seen:.4f} valF1={val['f1']:.4f}")
    return pd.DataFrame(history)


V8 EXTENSION — online PGD implementation


In [10]:
# ===== 9. Fixed-row evaluation shared by all methods and all seeds =====
section("COMMON FAIR EVALUATION")
eval_rng = np.random.RandomState(CONFIG["data_seed"] + 101)
eval_ids = eval_rng.choice(len(X_test_mi), min(CONFIG["evaluation_samples"], len(X_test_mi)), replace=False)
deep_ids = eval_ids[:min(CONFIG["deepfool_evaluation_samples"], len(eval_ids))]
square_ids = eval_ids[:min(CONFIG["square_evaluation_samples"], len(eval_ids))]

def evaluate_method(model, method, seed, X_full, feature_idx, bounds, mask):
    rows, sanity = [], []
    clean_metrics = classification_metrics(model, X_full, y_test)
    rows.append({"seed": seed, "method": method, "attack": "Clean", **clean_metrics})
    clean, labels = X_full[eval_ids], y_test[eval_ids]
    attack_specs = [
        ("FGSM", lambda: (fgsm_attack(model, clean, labels, CONFIG["attack_epsilon"], bounds, mask), None), CONFIG["attack_epsilon"]),
        ("PGD", lambda: (pgd_attack(model, clean, labels, CONFIG["attack_epsilon"], CONFIG["pgd_steps"], CONFIG["pgd_restarts"], bounds, mask, seed + 201), None), CONFIG["attack_epsilon"]),
    ]
    for name, generate, eps in attack_specs:
        started = time.perf_counter(); adv, queries = generate(); elapsed = time.perf_counter() - started
        stats = attack_sanity(model, clean, adv, labels, name, eps, queries); stats.update(method=method, seed=seed, attack_generation_seconds=elapsed)
        sanity.append(stats); rows.append({"seed": seed, "method": method, "attack": name, **classification_metrics(model, adv, labels)})

    dclean, dlabels = X_full[deep_ids], y_test[deep_ids]; started = time.perf_counter()
    dadv = raw_deepfool_attack(model, dclean, bounds); elapsed = time.perf_counter() - started
    stats = attack_sanity(model, dclean, dadv, dlabels, "DeepFool_raw", None); stats.update(method=method, seed=seed, attack_generation_seconds=elapsed)
    sanity.append(stats); rows.append({"seed": seed, "method": method, "attack": "DeepFool_raw", **classification_metrics(model, dadv, dlabels)})

    sclean, slabels = X_full[square_ids], y_test[square_ids]; started = time.perf_counter()
    sadv, queries = square_style_attack(model, sclean, slabels, CONFIG["attack_epsilon"], bounds, mask, seed + 301); elapsed = time.perf_counter() - started
    stats = attack_sanity(model, sclean, sadv, slabels, "Square-style_black-box", CONFIG["attack_epsilon"], queries)
    stats.update(method=method, seed=seed, attack_generation_seconds=elapsed); sanity.append(stats)
    rows.append({"seed": seed, "method": method, "attack": "Square-style_black-box", **classification_metrics(model, sadv, slabels)})
    return rows, sanity


COMMON FAIR EVALUATION

COMMON FAIR EVALUATION


In [12]:
# ===== 10. Train Baseline, Paper Defense, and V8 Defense for every declared seed =====
section("MULTI-SEED MAIN EXPERIMENT — [42, 123, 2024]")
X_train_ext, X_dev_ext, X_test_ext = X_train_bal[:, extension_idx], X_dev_mi[:, extension_idx], X_test_mi[:, extension_idx]
ext_bounds = (clip_min[extension_idx], clip_max[extension_idx]); ext_mask = structural_mask[extension_idx]
all_metric_rows, all_sanity_rows, timing_rows = [], [shap_development_stats] + static_sanity.copy(), []
models_by_seed, histories_by_seed = {}, {}

for seed in CONFIG["seeds"]:
    section(f"MAIN SEED {seed}")
    seed_dir = OUTPUT_DIR / "seed_runs" / str(seed); seed_dir.mkdir(parents=True, exist_ok=True)

    baseline = selector_model if seed == CONFIG["seeds"][0] else new_model(len(paper_mi_features), seed)
    started = time.perf_counter()
    baseline_history = selector_history.copy() if seed == CONFIG["seeds"][0] else train_clean(
        baseline, X_train_bal, y_train_bal, X_dev_mi, y_dev, CONFIG["baseline_epochs"], seed, f"Baseline seed={seed}")
    baseline_time = selector_training_seconds if seed == CONFIG["seeds"][0] else time.perf_counter() - started

    paper = new_model(len(paper_idx), seed + 10_000); started = time.perf_counter()
    paper_history = train_clean(paper, X_paper_static, y_paper_static, X_dev_paper, y_dev,
                                CONFIG["paper_defense_epochs"], seed + 10_000, f"Paper static defense seed={seed}")
    paper_time = time.perf_counter() - started

    v8 = new_model(len(extension_idx), seed + 20_000); started = time.perf_counter()
    v8_clean_history = train_clean(v8, X_train_ext, y_train_bal, X_dev_ext, y_dev,
                                   CONFIG["extension_clean_epochs"], seed + 20_000, f"V8 clean stage seed={seed}")
    online_started = time.perf_counter()
    v8_online_history = train_online_pgd(v8, X_train_ext, y_train_bal, X_dev_ext, y_dev, ext_bounds, ext_mask, seed + 30_000)
    online_time = time.perf_counter() - online_started; v8_time = time.perf_counter() - started

    for model, method, X_view, idx_view, bounds, mask in [
        (baseline, "Baseline", X_test_mi, list(range(len(paper_mi_features))), (clip_min, clip_max), structural_mask),
        (paper, "Paper Defense", X_test_paper, paper_idx, paper_bounds, paper_mask),
        (v8, "V8 Defense", X_test_ext, extension_idx, ext_bounds, ext_mask)]:
        rows, sanity = evaluate_method(model, method, seed, X_view, idx_view, bounds, mask)
        all_metric_rows.extend(rows); all_sanity_rows.extend(sanity)

    timing_rows.extend([
        {"seed": seed, "method": "Baseline", "feature_count": len(paper_mi_features), "training_epochs": CONFIG["baseline_epochs"], "training_rows_per_epoch": len(X_train_bal), "estimated_optimizer_updates": CONFIG["baseline_epochs"] * int(np.ceil(len(X_train_bal) / CONFIG["batch_size"])), "attack_training": "none", "attack_epsilon": np.nan, "attack_iterations": 0, "training_seconds": baseline_time},
        {"seed": seed, "method": "Paper Defense", "feature_count": len(paper_idx), "training_epochs": CONFIG["paper_defense_epochs"], "training_rows_per_epoch": len(X_paper_static), "estimated_optimizer_updates": CONFIG["paper_defense_epochs"] * int(np.ceil(len(X_paper_static) / CONFIG["batch_size"])), "attack_training": "static FGSM+PGD+raw DeepFool", "attack_epsilon": CONFIG["attack_epsilon"], "attack_iterations": CONFIG["pgd_steps"], "training_seconds": paper_time},
        {"seed": seed, "method": "V8 Defense", "feature_count": len(extension_idx), "training_epochs": CONFIG["extension_clean_epochs"] + CONFIG["extension_online_epochs"], "training_rows_per_epoch": len(X_train_ext), "estimated_optimizer_updates": (CONFIG["extension_clean_epochs"] + CONFIG["extension_online_epochs"]) * int(np.ceil(len(X_train_ext) / CONFIG["batch_size"])), "attack_training": "online current-model PGD", "attack_epsilon": CONFIG["attack_epsilon"], "attack_iterations": CONFIG["online_pgd_steps"], "training_seconds": v8_time, "online_training_seconds": online_time},
    ])
    baseline_history.to_csv(seed_dir / "training_history_baseline.csv", index=False)
    paper_history.to_csv(seed_dir / "training_history_paper.csv", index=False)
    pd.concat([v8_clean_history.assign(phase="clean"), v8_online_history.assign(phase="online_pgd")], ignore_index=True).to_csv(seed_dir / "training_history_v8.csv", index=False)
    torch.save(baseline.state_dict(), seed_dir / "baseline_model.pt"); torch.save(paper.state_dict(), seed_dir / "paper_defense_model.pt"); torch.save(v8.state_dict(), seed_dir / "v8_defense_model.pt")
    models_by_seed[seed] = {"Baseline": baseline, "Paper Defense": paper, "V8 Defense": v8}
    histories_by_seed[seed] = (baseline_history, paper_history, v8_clean_history, v8_online_history)

all_seed_results = pd.DataFrame(all_metric_rows)
attack_sanity_checks = pd.DataFrame(all_sanity_rows)
timing_results = pd.DataFrame(timing_rows)
all_seed_results.to_csv(OUTPUT_DIR / "all_seed_results.csv", index=False)
attack_sanity_checks.to_csv(OUTPUT_DIR / "attack_sanity_checks.csv", index=False)
attack_sanity_checks.to_csv(OUTPUT_DIR / "attack_perturbation_statistics.csv", index=False)
timing_results.to_csv(OUTPUT_DIR / "computational_overhead.csv", index=False)

metric_columns = ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "mcc", "roc_auc"]
mean_std_results = all_seed_results.groupby(["method", "attack"])[metric_columns].agg(["mean", "std", "min", "max"])
mean_std_results.columns = [f"{metric}_{stat}" for metric, stat in mean_std_results.columns]
mean_std_results.reset_index().to_csv(OUTPUT_DIR / "mean_std_results.csv", index=False)

main_seed = CONFIG["seeds"][0]; main_models = models_by_seed[main_seed]
shutil.copy2(OUTPUT_DIR / "seed_runs" / str(main_seed) / "baseline_model.pt", OUTPUT_DIR / "baseline_model.pt")
shutil.copy2(OUTPUT_DIR / "seed_runs" / str(main_seed) / "paper_defense_model.pt", OUTPUT_DIR / "paper_defense_model.pt")
shutil.copy2(OUTPUT_DIR / "seed_runs" / str(main_seed) / "v8_defense_model.pt", OUTPUT_DIR / "v8_defense_model.pt")
for method, filename in [("Baseline", "baseline_metrics.csv"), ("Paper Defense", "paper_defense_metrics.csv"), ("V8 Defense", "v8_metrics.csv")]:
    all_seed_results[all_seed_results.method == method].to_csv(OUTPUT_DIR / filename, index=False)
histories_by_seed[main_seed][0].to_csv(OUTPUT_DIR / "training_history_baseline.csv", index=False)
histories_by_seed[main_seed][1].to_csv(OUTPUT_DIR / "training_history_paper.csv", index=False)
pd.concat([histories_by_seed[main_seed][2].assign(phase="clean"), histories_by_seed[main_seed][3].assign(phase="online_pgd")], ignore_index=True).to_csv(OUTPUT_DIR / "training_history_v8.csv", index=False)


MULTI-SEED MAIN EXPERIMENT — [42, 123, 2024]

MAIN SEED 42


Paper static defense seed=42:   0%|          | 0/50 [00:00<?, ?it/s]

F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:33.)
  return data.pin_memory(device)


Paper static defense seed=42: epoch=1/50 val_f1=0.8413


F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:33.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm

...

Square-style_black-box SANITY | generated=1000 changed=94.1% meanL0=11.74 meanL2=0.1356 meanLinf=0.0470 maxLinf=0.0500 within=100.0 ASR=0.2125
FGSM SANITY | generated=2000 changed=100.0% meanL0=10.49 meanL2=0.1598 meanLinf=0.0500 maxLinf=0.0500 within=100.0 ASR=0.0033
PGD SANITY | generated=2000 changed=100.0% meanL0=10.98 meanL2=0.1474 meanLinf=0.0500 maxLinf=0.0500 within=100.0 ASR=0.0163


DeepFool:   0%|          | 0/500 [00:00<?, ?it/s]

DeepFool_raw SANITY | generated=500 changed=100.0% meanL0=12.00 meanL2=6.6142 meanLinf=4.0839 maxLinf=37.2475 within=nan ASR=0.8174


Square-style:   0%|          | 0/300 [00:00<?, ?it/s]

Square-style_black-box SANITY | generated=1000 changed=90.0% meanL0=9.38 meanL2=0.1438 meanLinf=0.0450 maxLinf=0.0500 within=100.0 ASR=0.0813


In [14]:
# ===== 11. Gradient-masking diagnostics: strength, epsilon, gradients, boundary coverage =====
section("V8 ADDITIONAL EXPERIMENT — gradient-masking hypothesis tests")
diagnostic_model = main_models["V8 Defense"]
diag_ids = eval_ids[:min(CONFIG["diagnostic_samples"], len(eval_ids))]
X_diag, y_diag = X_test_ext[diag_ids], y_test[diag_ids]

pgd_strength_rows = []
for steps in CONFIG["pgd_strength_steps"]:
    adv = pgd_attack(diagnostic_model, X_diag, y_diag, CONFIG["attack_epsilon"], steps,
                     CONFIG["diagnostic_restarts"], ext_bounds, ext_mask, main_seed + steps)
    stats = attack_sanity(diagnostic_model, X_diag, adv, y_diag, f"PGD-{steps}-R{CONFIG['diagnostic_restarts']}", CONFIG["attack_epsilon"])
    pgd_strength_rows.append({"steps": steps, "restarts": CONFIG["diagnostic_restarts"], **stats})
pgd_strength = pd.DataFrame(pgd_strength_rows); pgd_strength.to_csv(OUTPUT_DIR / "pgd_strength_sweep.csv", index=False)

epsilon_rows = []
for epsilon in CONFIG["epsilon_sweep"]:
    adv = pgd_attack(diagnostic_model, X_diag, y_diag, epsilon, 50, CONFIG["diagnostic_restarts"],
                     ext_bounds, ext_mask, main_seed + int(epsilon * 10000))
    stats = attack_sanity(diagnostic_model, X_diag, adv, y_diag, f"PGD-50-eps-{epsilon}", epsilon)
    epsilon_rows.append({"epsilon": epsilon, "steps": 50, "restarts": CONFIG["diagnostic_restarts"], **stats})
epsilon_sweep = pd.DataFrame(epsilon_rows); epsilon_sweep.to_csv(OUTPUT_DIR / "epsilon_sweep.csv", index=False)

diagnostic_model.eval(); xg = torch.as_tensor(X_diag, dtype=torch.float32, device=DEVICE).requires_grad_(True)
yg = torch.as_tensor(y_diag, dtype=torch.long, device=DEVICE); grad = torch.autograd.grad(F.cross_entropy(diagnostic_model(xg), yg), xg)[0]
grad_norm = grad.detach().norm(p=2, dim=1).cpu().numpy()
gradient_statistics = pd.DataFrame([{"n": len(grad_norm), "mean": grad_norm.mean(), "median": np.median(grad_norm),
                                     "p05": np.percentile(grad_norm, 5), "p25": np.percentile(grad_norm, 25),
                                     "p75": np.percentile(grad_norm, 75), "p95": np.percentile(grad_norm, 95),
                                     "near_zero_pct": 100 * (grad_norm < 1e-10).mean()}])
gradient_statistics.to_csv(OUTPUT_DIR / "gradient_statistics.csv", index=False)

def boundary_walk(model, X, y, eps, bounds, mask, seed):
    # Custom label-only diagnostic. Failed searches are omitted from adversarial arrays and counted.
    rng = np.random.RandomState(seed); clean_pred = predict_logits(model, X).argmax(1)
    valid_adv, valid_y, valid_clean, records = [], [], [], []
    mutable = np.flatnonzero(mask)
    for i in tqdm(range(len(X)), desc="Boundary-Walk diagnostic"):
        attempted = clean_pred[i] == y[i]; initialized = refined = False; queries = 1; candidate = None
        if attempted:
            trials = rng.uniform(-eps, eps, size=(CONFIG["boundary_start_trials"], X.shape[1])).astype(np.float32)
            trials[:, mask == 0] = 0
            row_lo, row_hi = np.minimum(bounds[0], X[i]), np.maximum(bounds[1], X[i])
            candidates = np.clip(X[i] + trials, row_lo, row_hi); candidates[:, mask == 0] = X[i, mask == 0]
            queries += len(candidates)
            pred = predict_logits(model, candidates).argmax(1); hits = np.flatnonzero(pred != y[i])
            if len(hits):
                initialized = True; high = candidates[hits[np.argmin(np.linalg.norm(candidates[hits] - X[i], axis=1))]].copy(); low = X[i].copy()
                for _ in range(CONFIG["boundary_refinement_steps"]):
                    mid = (low + high) / 2; queries += 1
                    if predict_logits(model, mid[None]).argmax(1)[0] != y[i]: high = mid
                    else: low = mid
                candidate = high; refined = predict_logits(model, candidate[None]).argmax(1)[0] != y[i]; queries += 1
        status = "refined" if refined else "initialized_only" if initialized else "failed" if attempted else "not_initially_correct"
        records.append({"sample": i, "attempted": attempted, "initialized": initialized, "refined": refined, "status": status, "queries": queries})
        if refined: valid_clean.append(X[i]); valid_adv.append(candidate); valid_y.append(y[i])
    return np.asarray(valid_clean, np.float32), np.asarray(valid_adv, np.float32), np.asarray(valid_y, np.int64), pd.DataFrame(records)

X_bclean, X_badv, y_bvalid, boundary_records = boundary_walk(
    diagnostic_model, X_test_ext[eval_ids[:CONFIG["boundary_samples"]]], y_test[eval_ids[:CONFIG["boundary_samples"]]],
    CONFIG["attack_epsilon"], ext_bounds, ext_mask, main_seed + 501)
boundary_records.to_csv(OUTPUT_DIR / "boundary_walk_coverage.csv", index=False)
boundary_summary = {"attempted": int(boundary_records.attempted.sum()), "initialized": int(boundary_records.initialized.sum()),
                    "refined": int(boundary_records.refined.sum()), "failed": int((boundary_records.status == "failed").sum())}
if len(X_badv):
    boundary_valid_stats = attack_sanity(diagnostic_model, X_bclean, X_badv, y_bvalid, "Boundary-Walk_valid_only", CONFIG["attack_epsilon"])
else: boundary_valid_stats = {"attack_success_rate": np.nan, "adversarial_accuracy": np.nan}
boundary_valid_stats.update({"method": "V8 Diagnostics", "seed": main_seed, "phase": "boundary_valid_only"})
print("Boundary Search Coverage:", boundary_summary)

# Same rows as the PGD diagnostic, for a meaningful white-box/gradient-free comparison.
diag_square_adv, diag_square_queries = square_style_attack(
    diagnostic_model, X_diag, y_diag, CONFIG["attack_epsilon"], ext_bounds, ext_mask, main_seed + 502)
diagnostic_square_stats = attack_sanity(
    diagnostic_model, X_diag, diag_square_adv, y_diag, "Diagnostic_Square-style_same_rows",
    CONFIG["attack_epsilon"], diag_square_queries)
diagnostic_square_stats.update({"method": "V8 Diagnostics", "seed": main_seed, "phase": "same_row_gradient_free"})
pd.DataFrame([diagnostic_square_stats]).to_csv(OUTPUT_DIR / "gradient_free_diagnostic.csv", index=False)


V8 ADDITIONAL EXPERIMENT — gradient-masking hypothesis tests
PGD-20-R5 SANITY | generated=500 changed=100.0% meanL0=10.87 meanL2=0.1508 meanLinf=0.0500 maxLinf=0.0500 within=100.0 ASR=0.0239
PGD-50-R5 SANITY | generated=500 changed=100.0% meanL0=10.78 meanL2=0.1524 meanLinf=0.0500 maxLinf=0.0500 within=100.0 ASR=0.0239
PGD-100-R5 SANITY | generated=500 changed=100.0% meanL0=10.77 meanL2=0.1524 meanLinf=0.0500 maxLinf=0.0500 within=100.0 ASR=0.0239
PGD-200-R5 SANITY | generated=500 changed=100.0% meanL0=10.81 meanL2=0.1528 meanLinf=0.0500 maxLinf=0.0500 within=100.0 ASR=0.0239
PGD-50-eps-0.01 SANITY | generated=500 changed=100.0% meanL0=10.37 meanL2=0.0311 meanLinf=0.0100 maxLinf=0.0100 within=100.0 ASR=0.0000
PGD-50-eps-0.025 SANITY | generated=500 changed=100.0% meanL0=10.51 meanL2=0.0774 meanLinf=0.0250 maxLinf=0.0250 within=100.0 ASR=0.0022
PGD-50-eps-0.05 SANITY | generated=500 changed=100.0% meanL0=10.80 meanL2=0.1530 meanLinf=0.0500 maxLinf=0.0500 within=100.0 ASR=0.0239
PGD-50-

Boundary-Walk diagnostic:   0%|          | 0/200 [00:00<?, ?it/s]

Boundary-Walk_valid_only SANITY | generated=1 changed=100.0% meanL0=6.00 meanL2=0.0673 meanLinf=0.0464 maxLinf=0.0464 within=100.0 ASR=1.0000
Boundary Search Coverage: {'attempted': 185, 'initialized': 1, 'refined': 1, 'failed': 184}


Square-style:   0%|          | 0/300 [00:00<?, ?it/s]

Diagnostic_Square-style_same_rows SANITY | generated=500 changed=89.6% meanL0=9.30 meanL2=0.1425 meanLinf=0.0448 maxLinf=0.0500 within=100.0 ASR=0.0239

V8 ADDITIONAL EXPERIMENT — gradient-masking hypothesis tests
PGD-20-R5 SANITY | generated=500 changed=100.0% meanL0=10.87 meanL2=0.1508 meanLinf=0.0500 maxLinf=0.0500 within=100.0 ASR=0.0239
PGD-50-R5 SANITY | generated=500 changed=100.0% meanL0=10.78 meanL2=0.1524 meanLinf=0.0500 maxLinf=0.0500 within=100.0 ASR=0.0239
PGD-100-R5 SANITY | generated=500 changed=100.0% meanL0=10.77 meanL2=0.1524 meanLinf=0.0500 maxLinf=0.0500 within=100.0 ASR=0.0239
PGD-200-R5 SANITY | generated=500 changed=100.0% meanL0=10.81 meanL2=0.1528 meanLinf=0.0500 maxLinf=0.0500 within=100.0 ASR=0.0239
PGD-50-eps-0.01 SANITY | generated=500 changed=100.0% meanL0=10.37 meanL2=0.0311 meanLinf=0.0100 maxLinf=0.0100 within=100.0 ASR=0.0000
PGD-50-eps-0.025 SANITY | generated=500 changed=100.0% meanL0=10.51 meanL2=0.0774 meanLinf=0.0250 maxLinf=0.0250 within=100.0 AS

Boundary-Walk diagnostic:   0%|          | 0/200 [00:00<?, ?it/s]

Boundary-Walk_valid_only SANITY | generated=1 changed=100.0% meanL0=6.00 meanL2=0.0673 meanLinf=0.0464 maxLinf=0.0464 within=100.0 ASR=1.0000
Boundary Search Coverage: {'attempted': 185, 'initialized': 1, 'refined': 1, 'failed': 184}


Square-style:   0%|          | 0/300 [00:00<?, ?it/s]

Diagnostic_Square-style_same_rows SANITY | generated=500 changed=89.6% meanL0=9.30 meanL2=0.1425 meanLinf=0.0448 maxLinf=0.0500 within=100.0 ASR=0.0239


In [16]:
# ===== 12. Independent-surrogate transfer attack =====
section("V8 ADDITIONAL EXPERIMENT — transfer attack")
surrogate = new_model(len(paper_mi_features), main_seed + 90_000)
surrogate_history = train_clean(surrogate, X_train_bal, y_train_bal, X_dev_mi, y_dev,
                                CONFIG["surrogate_epochs"], main_seed + 90_000, "Independent surrogate")
transfer_ids = eval_ids[:min(CONFIG["transfer_samples"], len(eval_ids))]
X_transfer_clean, y_transfer = X_test_mi[transfer_ids], y_test[transfer_ids]
X_transfer_adv = pgd_attack(surrogate, X_transfer_clean, y_transfer, CONFIG["attack_epsilon"], 50,
                            CONFIG["diagnostic_restarts"], (clip_min, clip_max), structural_mask, main_seed + 601)
transfer_source_stats = attack_sanity(
    surrogate, X_transfer_clean, X_transfer_adv, y_transfer, "Surrogate_PGD_transfer_source", CONFIG["attack_epsilon"])
transfer_source_stats.update({"method": "Transfer Source", "seed": main_seed, "phase": "transfer_generation"})
transfer_rows = []
for method, model, idx in [("Baseline", main_models["Baseline"], list(range(len(paper_mi_features)))),
                           ("Paper Defense", main_models["Paper Defense"], paper_idx),
                           ("V8 Defense", main_models["V8 Defense"], extension_idx)]:
    clean_view, adv_view = X_transfer_clean[:, idx], X_transfer_adv[:, idx]
    clean_pred, adv_pred = predict_logits(model, clean_view).argmax(1), predict_logits(model, adv_view).argmax(1)
    correct = clean_pred == y_transfer
    transfer_rows.append({"target": method, "source": "independent IDSNet", "source_feature_count": len(paper_mi_features),
                          "target_feature_count": len(idx), "n": len(y_transfer), "clean_accuracy": correct.mean(),
                          "transfer_accuracy": (adv_pred == y_transfer).mean(),
                          "transfer_asr": ((adv_pred != y_transfer) & correct).sum() / max(correct.sum(), 1)})
transfer_results = pd.DataFrame(transfer_rows); transfer_results.to_csv(OUTPUT_DIR / "transfer_results.csv", index=False)
surrogate_history.to_csv(OUTPUT_DIR / "training_history_surrogate.csv", index=False)


V8 ADDITIONAL EXPERIMENT — transfer attack


Independent surrogate:   0%|          | 0/30 [00:00<?, ?it/s]

F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:33.)
  return data.pin_memory(device)


Independent surrogate: epoch=1/30 val_f1=0.9083


F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:33.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm

...

Independent surrogate: epoch=20/30 val_f1=0.9163


F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:33.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm

Independent surrogate: epoch=25/30 val_f1=0.9128


F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:33.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm_native\rocm_env\Lib\site-packages\torch\utils\data\_utils\pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at C:\b\pytorch\aten\src\ATen\native\Memory.cpp:48.)
  return data.pin_memory(device)
F:\PythonProjects\aml_rocm

Independent surrogate: epoch=30/30 val_f1=0.9157
Surrogate_PGD_transfer_source SANITY | generated=1000 changed=100.0% meanL0=36.60 meanL2=0.2590 meanLinf=0.0500 maxLinf=0.0500 within=100.0 ASR=0.1323


In [18]:
# ===== 13. Trade-offs, overhead, masking conclusion, plots, and final package =====
section("FINAL ANALYSIS AND THESIS ARTIFACTS")
diagnostic_sanity = pd.concat([
    pgd_strength.assign(method="V8 Diagnostics", seed=main_seed, phase="pgd_strength"),
    epsilon_sweep.assign(method="V8 Diagnostics", seed=main_seed, phase="epsilon_sweep"),
    pd.DataFrame([diagnostic_square_stats, boundary_valid_stats, transfer_source_stats]),
], ignore_index=True, sort=False)
attack_sanity_checks = pd.concat([attack_sanity_checks, diagnostic_sanity], ignore_index=True, sort=False)
attack_sanity_checks.to_csv(OUTPUT_DIR / "attack_sanity_checks.csv", index=False)
attack_sanity_checks.to_csv(OUTPUT_DIR / "attack_perturbation_statistics.csv", index=False)
main = all_seed_results[all_seed_results.seed == main_seed].copy()
pivot = main.pivot(index="method", columns="attack", values="accuracy")
tradeoff_rows = []
baseline_clean = pivot.loc["Baseline", "Clean"]
clean_rows = main[main.attack == "Clean"].set_index("method")
for method in pivot.index:
    attacked = [c for c in ["FGSM", "PGD", "DeepFool_raw", "Square-style_black-box"] if c in pivot.columns]
    tradeoff_rows.append({"method": method, "clean_accuracy": pivot.loc[method, "Clean"],
                          "clean_accuracy_delta_vs_baseline": pivot.loc[method, "Clean"] - baseline_clean,
                          "clean_f1": clean_rows.loc[method, "f1"],
                          "clean_f1_delta_vs_baseline": clean_rows.loc[method, "f1"] - clean_rows.loc["Baseline", "f1"],
                          "clean_precision": clean_rows.loc[method, "precision"],
                          "clean_precision_delta_vs_baseline": clean_rows.loc[method, "precision"] - clean_rows.loc["Baseline", "precision"],
                          "clean_recall": clean_rows.loc[method, "recall"],
                          "clean_recall_delta_vs_baseline": clean_rows.loc[method, "recall"] - clean_rows.loc["Baseline", "recall"],
                          "mean_attack_accuracy": pivot.loc[method, attacked].mean(),
                          "mean_robustness_gain_vs_baseline": (pivot.loc[method, attacked] - pivot.loc["Baseline", attacked]).mean(),
                          **{f"{attack}_accuracy_gain_vs_baseline": pivot.loc[method, attack] - pivot.loc["Baseline", attack]
                             for attack in attacked}})
tradeoff = pd.DataFrame(tradeoff_rows); tradeoff.to_csv(OUTPUT_DIR / "clean_robust_tradeoff.csv", index=False)

baseline_time_mean = timing_results[timing_results.method == "Baseline"].training_seconds.mean()
timing_results["retraining_overhead_pct_vs_baseline"] = 100 * (timing_results.training_seconds / baseline_time_mean - 1)
timing_results["preprocessing_seconds_shared"] = preprocessing_time
timing_results["feature_selection_seconds_shared"] = feature_selection_time
timing_results["paper_static_attack_generation_seconds_shared"] = paper_attack_generation_time
timing_results["evaluation_attack_generation_seconds"] = timing_results.apply(
    lambda row: attack_sanity_checks[(attack_sanity_checks.seed == row.seed) &
                                     (attack_sanity_checks.method == row.method) &
                                     (attack_sanity_checks.phase.isna() if "phase" in attack_sanity_checks.columns else True)]
                .attack_generation_seconds.fillna(0).sum(), axis=1)
timing_results["total_pipeline_seconds_estimate"] = (
    timing_results.training_seconds + preprocessing_time + timing_results.evaluation_attack_generation_seconds +
    np.where(timing_results.method == "Baseline", 0, feature_selection_time) +
    np.where(timing_results.method == "Paper Defense",
             paper_attack_generation_time + paper_generator_training_seconds, 0))
timing_results.to_csv(OUTPUT_DIR / "computational_overhead.csv", index=False)

# Inference latency uses the same fixed subset and synchronization where supported.
latency_rows = []
for method, model, X_view in [("Baseline", main_models["Baseline"], X_test_mi[eval_ids]),
                              ("Paper Defense", main_models["Paper Defense"], X_test_paper[eval_ids]),
                              ("V8 Defense", main_models["V8 Defense"], X_test_ext[eval_ids])]:
    for _ in range(3): predict_logits(model, X_view[:min(128, len(X_view))])
    if torch.cuda.is_available(): torch.cuda.synchronize()
    started = time.perf_counter(); predict_logits(model, X_view)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    latency_rows.append({"method": method, "samples": len(X_view), "total_seconds": elapsed, "milliseconds_per_sample": 1000 * elapsed / len(X_view)})
pd.DataFrame(latency_rows).to_csv(OUTPUT_DIR / "inference_time.csv", index=False)
method_comparison = (tradeoff.merge(timing_results.groupby("method", as_index=False).agg(
    feature_count=("feature_count", "first"), training_epochs=("training_epochs", "first"),
    estimated_optimizer_updates=("estimated_optimizer_updates", "first"),
    attack_training=("attack_training", "first"), mean_training_seconds=("training_seconds", "mean"),
    mean_total_pipeline_seconds=("total_pipeline_seconds_estimate", "mean"),
    mean_training_overhead_pct=("retraining_overhead_pct_vs_baseline", "mean")), on="method")
    .merge(pd.DataFrame(latency_rows)[["method", "milliseconds_per_sample"]], on="method"))
for attack in pivot.columns:
    method_comparison[f"{attack}_accuracy_main_seed"] = method_comparison.method.map(pivot[attack])
method_comparison.to_csv(OUTPUT_DIR / "method_comparison.csv", index=False)

strength_accuracy = pgd_strength.set_index("steps").adversarial_accuracy
masking_flags = {
    "pgd_accuracy_increases_with_strength": bool(strength_accuracy.loc[200] > strength_accuracy.loc[20] + 0.02),
    "near_zero_gradients_over_50pct": bool(gradient_statistics.iloc[0].near_zero_pct > 50),
}
v8_main_pgd = float(strength_accuracy.loc[200]); v8_square = float(diagnostic_square_stats["adversarial_accuracy"])
masking_flags["black_box_over_10pp_stronger_than_pgd"] = v8_square < v8_main_pgd - 0.10
v8_transfer = float(transfer_results[transfer_results.target == "V8 Defense"].transfer_accuracy.iloc[0])
masking_flags["transfer_over_10pp_stronger_than_pgd"] = v8_transfer < v8_main_pgd - 0.10
flag_count = sum(masking_flags.values())
masking_conclusion = "NO EVIDENCE" if flag_count == 0 else "POSSIBLE EVIDENCE" if flag_count == 1 else "STRONG EVIDENCE"
masking_note = "This rule-based synthesis is a diagnostic conclusion, not a proof of absence/presence."

ablation_results = tradeoff.copy()
ablation_results["interpretation"] = ["fixed MI, clean training", "fixed SHAP, static multi-attack training", "adaptive SHAP, online PGD training"]
ablation_results.to_csv(OUTPUT_DIR / "ablation_results.csv", index=False)

plt.style.use("seaborn-v0_8-whitegrid")
def savefig(name): plt.tight_layout(); plt.savefig(OUTPUT_DIR / name, dpi=180, bbox_inches="tight"); plt.close()
pivot.T.plot(kind="bar", figsize=(11, 5)); plt.ylabel("Accuracy"); plt.ylim(0, 1); plt.title("Clean and adversarial accuracy — main seed"); savefig("attack_accuracy_comparison.png")
asr_plot = attack_sanity_checks[(attack_sanity_checks.seed == main_seed) &
                                (attack_sanity_checks.method.isin(["Baseline", "Paper Defense", "V8 Defense"]))].pivot_table(
                                    index="attack", columns="method", values="attack_success_rate")
asr_plot.plot(kind="bar", figsize=(11, 5)); plt.ylabel("Attack success rate"); plt.ylim(0, 1); savefig("attack_success_comparison.png")
plt.plot(epsilon_sweep.epsilon, epsilon_sweep.adversarial_accuracy, marker="o"); plt.xlabel("L∞ epsilon (standardized space)"); plt.ylabel("Accuracy"); savefig("epsilon_vs_accuracy.png")
plt.plot(pgd_strength.steps, pgd_strength.adversarial_accuracy, marker="o"); plt.xlabel("PGD iterations (5 restarts)"); plt.ylabel("Accuracy"); savefig("pgd_strength_curve.png")
for _, row in tradeoff.iterrows(): plt.scatter(row.clean_accuracy, row.mean_attack_accuracy, label=row.method, s=70)
plt.xlabel("Clean accuracy"); plt.ylabel("Mean attack accuracy"); plt.legend(); savefig("clean_vs_robust_tradeoff.png")
shap_scores.sort_values("paper_stability_clipped").tail(20).plot.barh(x="feature", y="paper_stability_clipped", figsize=(9, 7), legend=False); savefig("feature_stability_plot.png")
shap_scores.sort_values("clean_importance").tail(20).plot.barh(x="feature", y="clean_importance", figsize=(9, 7), legend=False); savefig("shap_feature_importance.png")
timing_results.groupby("method").training_seconds.mean().plot.bar(figsize=(8, 5)); plt.ylabel("Training seconds"); savefig("overhead_comparison.png")

final_summary = {
    "experiment": CONFIG["experiment_name"], "paper_reproduction_exact": False,
    "paper_mi_features": len(paper_mi_features), "paper_shap_features": len(paper_shap_features),
    "v8_extension_features": len(extension_features), "seeds": CONFIG["seeds"],
    "main_seed_method_comparison": method_comparison.to_dict(orient="records"),
    "gradient_masking_conclusion": masking_conclusion, "gradient_masking_flags": masking_flags,
    "gradient_masking_note": masking_note, "boundary_coverage": boundary_summary,
    "limitations": ["Standardized feature-space perturbations are not guaranteed traffic-feasible.",
                    "The paper omits several hyperparameters; declared assumptions prevent exact reproduction.",
                    "Square-style and Boundary-Walk are custom tabular diagnostics, not canonical implementations.",
                    "Raw DeepFool is not directly epsilon-comparable to FGSM/PGD."],
}
with open(OUTPUT_DIR / "final_summary.json", "w", encoding="utf-8") as f: json.dump(final_summary, f, indent=2)
pd.json_normalize(final_summary).to_csv(OUTPUT_DIR / "final_summary.csv", index=False)

expected_pairs = {(seed, method, attack) for seed in CONFIG["seeds"]
                  for method in ["Baseline", "Paper Defense", "V8 Defense"]
                  for attack in ["Clean", "FGSM", "PGD", "DeepFool_raw", "Square-style_black-box"]}
observed_pairs = set(all_seed_results[["seed", "method", "attack"]].itertuples(index=False, name=None))
bounded_sanity = attack_sanity_checks[~attack_sanity_checks.attack.str.contains("DeepFool", case=False, na=False)]
self_audit = {
    "no_test_leakage_by_construction": True,
    "scaler_encoder_mi_smote_fit_on_train_only": True,
    "shap_uses_development_not_test": True,
    "online_attacks_generated_inside_each_batch": True,
    "online_attacks_regenerated_each_epoch": bool(all(h[3].attacks_regenerated.all() for h in histories_by_seed.values())),
    "no_missing_seed_method_attack_rows": expected_pairs == observed_pairs,
    "bounded_attacks_within_epsilon": bool((bounded_sanity.within_epsilon_pct.dropna() >= 100 - 1e-9).all()),
    "raw_deepfool_separately_named": "DeepFool_raw" in set(all_seed_results.attack),
    "custom_square_separately_named": "Square-style_black-box" in set(all_seed_results.attack),
    "boundary_failures_reported": sum(boundary_summary.values()) > 0 and "failed" in boundary_summary,
    "three_declared_seeds_evaluated": set(all_seed_results.seed) == set(CONFIG["seeds"]),
    "gradient_masking_is_result_derived": masking_conclusion in {"NO EVIDENCE", "POSSIBLE EVIDENCE", "STRONG EVIDENCE"},
}
if not all(self_audit.values()):
    raise RuntimeError(f"Final scientific self-audit failed: {self_audit}")
with open(OUTPUT_DIR / "self_audit.json", "w", encoding="utf-8") as f: json.dump(self_audit, f, indent=2)

required = ["config.json", "environment.json", "data_summary.csv", "mi_scores.csv", "shap_stability_scores.csv",
            "selected_features_mi.json", "selected_features_shap.json", "baseline_model.pt", "paper_defense_model.pt",
            "v8_defense_model.pt", "baseline_metrics.csv", "paper_defense_metrics.csv", "v8_metrics.csv",
            "all_seed_results.csv", "mean_std_results.csv", "attack_sanity_checks.csv", "pgd_strength_sweep.csv",
            "epsilon_sweep.csv", "gradient_statistics.csv", "transfer_results.csv", "ablation_results.csv",
            "training_history_baseline.csv", "training_history_paper.csv", "training_history_v8.csv",
            "attack_accuracy_comparison.png", "attack_success_comparison.png", "epsilon_vs_accuracy.png",
            "pgd_strength_curve.png", "clean_vs_robust_tradeoff.png", "overhead_comparison.png",
            "feature_stability_plot.png", "shap_feature_importance.png", "final_summary.json", "final_summary.csv"]
missing = [name for name in required if not (OUTPUT_DIR / name).exists()]
if missing: raise RuntimeError(f"Final artifact validation failed; missing: {missing}")
archive = shutil.make_archive("v8_results", "zip", root_dir=OUTPUT_DIR)

print("\n" + "=" * 88 + "\nV8 FINAL EXPERIMENT SUMMARY\n" + "=" * 88)
print(f"MI features={len(paper_mi_features)} | paper SHAP={len(paper_shap_features)} | V8 adaptive={len(extension_features)}")
print(mean_std_results.reset_index().to_string(index=False))
print("\nClean/robust trade-off:\n", tradeoff.to_string(index=False))
print("\nBoundary coverage:", boundary_summary)
print(f"\nGradient masking conclusion: {masking_conclusion}. {masking_note}")
print("\nSelf-audit:", json.dumps(self_audit, indent=2))
print(f"Artifacts validated: {len(required)} | archive={archive}")


FINAL ANALYSIS AND THESIS ARTIFACTS

V8 FINAL EXPERIMENT SUMMARY
MI features=40 | paper SHAP=15 | V8 adaptive=12
       method                 attack  accuracy_mean  accuracy_std  accuracy_min  accuracy_max  balanced_accuracy_mean  balanced_accuracy_std  balanced_accuracy_min  balanced_accuracy_max  precision_mean  precision_std  precision_min  precision_max  recall_mean  recall_std  recall_min  recall_max  f1_mean   f1_std   f1_min   f1_max  mcc_mean  mcc_std   mcc_min   mcc_max  roc_auc_mean  roc_auc_std  roc_auc_min  roc_auc_max
     Baseline                  Clean       0.967201      0.003564      0.964301      0.971180                0.947890               0.001671               0.946310               0.949639        0.928239       0.017632       0.914583       0.948144     0.914620    0.001847    0.912527    0.916020 0.921313 0.007786 0.914949 0.929995  0.900689 0.010241  0.892358  0.912123      0.980430     0.000170     0.980307     0.980624
     Baseline           DeepFool_raw

## Final interpretation boundary

This notebook can support claims about clean performance, robustness to the explicitly configured
feature-space attacks, multi-seed stability, and compute trade-offs on the fixed CSE-CIC-IDS2018
split. It does **not** establish cross-dataset generalization, real zero-day detection, or physical
network-flow feasibility. Any gradient-masking conclusion is diagnostic and must follow the printed
evidence category rather than being assumed in advance.